# Inference Configuration
Specify
- path

In [2]:
path = "../../output/protenn2/v5"


In [3]:
import json
import os.path
import pickle

import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from src.protenn2.utils import get_train_val_test_paths, get_device

# get file paths

log_path = os.path.join(path, "log")

label_encoder_path = os.path.join(path, "label_encoder.pkl")
model_path = os.path.join(path, "best_model.pt")
with open(os.path.join(path, "params.json"), "r") as f:
    params = json.load(f)
if "input_folder" not in params:
    raise ValueError("input_folder must be specified")
dataset_path = os.path.join("../../", params["input_folder"])
train_path, val_path, test_path = get_train_val_test_paths(dataset_path)


In [4]:
from src.protenn2.utils import calculate_max_protein_length
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.model import CathPredEnn2
from src.protenn2.analysis.cath_hierarchy_mapper import CATHHierarchyMapper

# Initialize objects

device = get_device()
with open(label_encoder_path, "rb") as f:
    label_encoder: LabelEncoder = pickle.load(f)
num_classes = len(label_encoder.classes_)
max_protein_length = calculate_max_protein_length(dataset_path)

model = CathPredEnn2(num_classes=num_classes)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
test_dataset = CathPredPerResidueDataset(test_path, label_encoder=label_encoder,
                                         embedding_dir="../../data/embeddings/protein_embeddings_new")

collate_fn = create_protein_collate_fn(max_protein_length, test_dataset.padding_encoded_id)

test_dataloader = DataLoader(test_dataset, collate_fn=collate_fn)

mapper = CATHHierarchyMapper(label_encoder=label_encoder)

Using MPS (Apple Silicon GPU).
Max protein length: 599
Dataset initialized with 1317 unique proteins.


In [5]:
from src.protenn2.analysis.inference import run_inference

y_true_labels_list, y_pred_confidences_list, protein_chain_id_list = run_inference(model=model,
                                                                                   dataloader=test_dataloader,
                                                                                   padding_encoded_id=test_dataset.padding_encoded_id,
                                                                                   device=device,
                                                                                   return_protein_chain_id=True)

Running inference on 1317 proteins...


Inference Progress: 100%|██████████| 1317/1317 [00:05<00:00, 261.60it/s]

Inference complete. Processed 1317 proteins


# Analysis Configuration

In [6]:
from src.protenn2.utils import call_domains_list

bootstrap_samples = 1000
post_process_kwargs = {"reporting_threshold": 0.1, "region_min_length": 20, "gaussian_sigma": 2}
post_process_func = call_domains_list
metrics_to_compute = ("accuracy", "f1_score", "jaccard_score", "recall_score", "precision_score",
                      "segment_overlap_score")

# metrics_to_compute = ("f1_score")

In [7]:
from src.protenn2.analysis.metrics import calculate_metrics_for_cath_levels

all_results = calculate_metrics_for_cath_levels(y_true_labels_list=y_true_labels_list,
                                                y_pred_confidences_list=y_pred_confidences_list, mapper=mapper,
                                                bootstrap_samples=bootstrap_samples,
                                                post_process_func=post_process_func,
                                                post_process_kwargs=post_process_kwargs,
                                                metrics_to_compute=metrics_to_compute,
                                                levels=("D",))

---- Computing Metrics for hierarchy: D
('accuracy', 'f1_score', 'jaccard_score', 'recall_score', 'precision_score', 'segment_overlap_score')
----- Accuracy Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:01<00:00, 676.90it/s]


{'mean': np.float64(0.76242804873105), 'ci_lower': np.float64(0.7495992233620202), 'ci_upper': np.float64(0.7747200627775104), 'alpha': 0.05}
----- F1 Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:09<00:00, 110.85it/s]


{'mean': np.float64(0.8156244751178349), 'ci_lower': np.float64(0.8033828865324554), 'ci_upper': np.float64(0.8272405324685513), 'alpha': 0.05}
----- Jaccard Score Metrics -----


...

{'mean': np.float64(0.5906528230326235), 'ci_lower': np.float64(0.5681623356192369), 'ci_upper': np.float64(0.6133222389081318), 'alpha': 0.05}
----- Precision Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:09<00:00, 106.00it/s]


{'mean': np.float64(0.9226536184096017), 'ci_lower': np.float64(0.9105819463802421), 'ci_upper': np.float64(0.9342718249177431), 'alpha': 0.05}
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:51<00:00, 19.57it/s]

{'mean': np.float64(0.6294038376055673), 'ci_lower': np.float64(0.6043324647561835), 'ci_upper': np.float64(0.6539816066936107), 'alpha': 0.05}


# All results

In [10]:
all_results

{'raw_D': {'accuracy': {'mean': np.float64(0.76242804873105),
   'ci_lower': np.float64(0.7495992233620202),
   'ci_upper': np.float64(0.7747200627775104),
   'alpha': 0.05},
  'f1_score': {'mean': np.float64(0.8156244751178349),
   'ci_lower': np.float64(0.8033828865324554),
   'ci_upper': np.float64(0.8272405324685513),
   'alpha': 0.05},
  'jaccard_score': {'mean': np.float64(0.6886982597496886),
   'ci_lower': np.float64(0.6713784036266762),
   'ci_upper': np.float64(0.7053795389221053),
   'alpha': 0.05},
  'recall_score': {'mean': np.float64(0.7866925794848958),
   'ci_lower': np.float64(0.7708809483167132),
   'ci_upper': np.float64(0.8010449388135188),
   'alpha': 0.05},
  'precision_score': {'mean': np.float64(0.8468207615260459),
   'ci_lower': np.float64(0.8328174521007748),
   'ci_upper': np.float64(0.8610030341848022),
   'alpha': 0.05},
  'segment_overlap_score': {'mean': np.float64(0.5951108932402825),
   'ci_lower': np.float64(0.5727787989906207),
   'ci_upper': np.floa

In [11]:
with open(os.path.join(path, "metrics_test_D.json"), "w") as f:
    json.dump(all_results, f)